In [2]:
import pandas as pd
from pathlib import Path
rd = Path.cwd().parent/'data'/'raw/'
wd = Path.cwd().parent/'data'/'processed/'


In [3]:
df = pd.read_csv(rd/'02_nav_history.csv')
df['date']=pd.to_datetime(df['date'])
df['date']

0       2022-01-03
1       2022-01-04
2       2022-01-05
3       2022-01-06
4       2022-01-07
           ...    
45995   2026-05-25
45996   2026-05-26
45997   2026-05-27
45998   2026-05-28
45999   2026-05-29
Name: date, Length: 46000, dtype: datetime64[us]

In [4]:
df.sort_values(by=['amfi_code','date'])
df = df.drop_duplicates()
df = df[df['nav']>0]

In [5]:
df.set_index('date')

,amfi_code,nav
date,,
2022-01-03,119551,54.3856
2022-01-04,119551,54.3474
2022-01-05,119551,54.6869
2022-01-06,119551,55.4550
2022-01-07,119551,55.3692
...,...,...
2026-05-25,149324,292.4810
2026-05-26,149324,291.2707
2026-05-27,149324,288.8007


In [6]:
df2=[]
for amfi_code, group in df.groupby('amfi_code'):
        group = group.set_index('date')
        
        full_date_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
        
        group = group.reindex(full_date_range)
        group['amfi_code'] = amfi_code  # Populate the amfi_code for the newly created rows
        
        group['nav'] = group['nav'].ffill()
        
        group = group.reset_index().rename(columns={'index': 'date'})
        df2.append(group)

In [7]:
final_df = pd.concat(df2, ignore_index=True)
    
final_df = final_df.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)

In [8]:
final_df

,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639
...,...,...,...
64315,2026-05-25,149324,292.4810
64316,2026-05-26,149324,291.2707
64317,2026-05-27,149324,288.8007
64318,2026-05-28,149324,280.6873


In [9]:
final_df.to_csv(wd/'clean_nav.csv',index=False)

In [10]:
#2nd cleaning

df_it = pd.read_csv(rd/'08_investor_transactions.csv')
df_it['transaction_date'] = pd.to_datetime(df_it['transaction_date'])

In [11]:
df_it[df_it['amount_inr']<=0]
#all valid
df_it.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [12]:
df_it['transaction_type'].unique()

<StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str

In [13]:
df_it['kyc_status'].unique()

<StringArray>
['Verified', 'Pending']
Length: 2, dtype: str

In [14]:
df_it.to_csv(wd/'clean_transactions.csv',index=False)

In [15]:
#3rd cleaning
df_sp = pd.read_csv(rd/'07_scheme_performance.csv')
print(df_sp.head())
df_sp.dtypes

   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   benchmark_3yr_pct  alpha  beta  sharpe_ratio  sortino_ratio  \
0              11.49

amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object

In [16]:
df_sp[df_sp['sharpe_ratio']<=0]
#no sharpe is negative

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [17]:
df_sp[df_sp['expense_ratio_pct'] <0.1]
df_sp[df_sp['expense_ratio_pct'] >2.5]
# all in range

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [18]:
df_sp.to_csv(wd/'clean_performance.csv',index=False)

In [19]:
#4th cleaning
df_fm = pd.read_csv(rd/'01_fund_master.csv')
print(df_fm.head())
df_fm.dtypes

   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Plan - Growth   

  category sub_category     plan launch_date                  benchmark  \
0   Equity    Large Cap  Regular  2006-02-14              NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01              NIFTY 100 TRI   
2   Equity    Small Cap  Regular  2009-09-09       BSE 250 SmallCap TRI   
3   Equity    Small Cap   Direct  2013-01-01       BSE 250 SmallCap TRI   
4     Debt         Gilt  Regular  2000-12-30  CRISIL Dynamic Gilt Index   

   expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount  \

amfi_code               int64
fund_house                str
scheme_name               str
category                  str
sub_category              str
plan                      str
launch_date               str
benchmark                 str
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager              str
risk_category             str
sebi_category_code        str
dtype: object

In [20]:
df_fm = df_fm.drop_duplicates()

In [21]:
df_fm['launch_date']=pd.to_datetime(df_fm['launch_date'])

In [22]:
df_fm.dtypes

amfi_code                      int64
fund_house                       str
scheme_name                      str
category                         str
sub_category                     str
plan                             str
launch_date           datetime64[us]
benchmark                        str
expense_ratio_pct            float64
exit_load_pct                float64
min_sip_amount                 int64
min_lumpsum_amount             int64
fund_manager                     str
risk_category                    str
sebi_category_code               str
dtype: object

In [23]:
#validating at a glance
for cols in df_fm.columns:
    print(df_fm[cols].unique())

[119551 119552 119598 119599 119120 100016 125497 100033 125498 100025
 120503 120504 120505 120506 120507 118632 118633 118634 118635 118636
 120841 120842 120843 120844 119092 119093 119094 119095 101206 101207
 101208 102885 102886 102887 148567 148568 148569 149322 149323 149324]
<StringArray>
[         'SBI Mutual Fund',         'HDFC Mutual Fund',
      'ICICI Prudential MF',          'Nippon India MF',
        'Kotak Mahindra MF',         'Axis Mutual Fund',
 'Aditya Birla Sun Life MF',          'UTI Mutual Fund',
           'Mirae Asset MF',          'DSP Mutual Fund']
Length: 10, dtype: str
<StringArray>
[            'SBI Bluechip Fund - Regular Plan - Growth',
              'SBI Bluechip Fund - Direct Plan - Growth',
            'SBI Small Cap Fund - Regular Plan - Growth',
             'SBI Small Cap Fund - Direct Plan - Growth',
          'SBI Magnum Gilt Fund - Regular Plan - Growth',
             'HDFC Top 100 Fund - Regular Plan - Growth',
              'HDFC Top 100 Fun

In [24]:
df_fm.to_csv(wd/'clean_fund.csv',index=False)

In [25]:
#5th cleaning
df_fh = pd.read_csv(rd/'03_aum_by_fund_house.csv')
print(df_fh.head())
df_fh.dtypes

         date           fund_house  aum_lakh_crore  aum_crore  num_schemes
0  2022-03-31      SBI Mutual Fund            6.05     605000          186
1  2022-03-31  ICICI Prudential MF            4.65     465000          216
2  2022-03-31     HDFC Mutual Fund            4.35     435000          195
3  2022-03-31      Nippon India MF            2.70     270000          177
4  2022-03-31    Kotak Mahindra MF            2.70     270000          168


date                  str
fund_house            str
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64
dtype: object

In [26]:
#validating at a glance
for cols in df_fh.columns:
    print(df_fh[cols].unique())

<StringArray>
['2022-03-31', '2022-09-30', '2023-03-31', '2023-09-30', '2024-03-31',
 '2024-09-30', '2024-12-31', '2025-03-31', '2025-12-31']
Length: 9, dtype: str
<StringArray>
[         'SBI Mutual Fund',      'ICICI Prudential MF',
         'HDFC Mutual Fund',          'Nippon India MF',
        'Kotak Mahindra MF', 'Aditya Birla Sun Life MF',
         'Axis Mutual Fund',          'UTI Mutual Fund',
           'Mirae Asset MF',          'DSP Mutual Fund']
Length: 10, dtype: str
[ 6.05  4.65  4.35  2.7   2.78  2.5   2.3   1.05  1.1   6.3   4.88  4.45
  2.72  2.85  2.4   2.32  1.08  1.12  7.17  5.    4.5   2.93  2.89  2.75
  2.41  2.39  1.16  1.15  8.45  5.9   5.35  3.4   3.25  3.08  2.6   2.65
  1.42  1.32 10.    6.8   6.45  4.3   3.8   2.8   2.9   1.75  1.55 10.8
  7.42  7.1   4.68  4.05  3.62  1.9   1.72 11.14  8.74  7.87  5.7   4.89
  3.84  3.    3.52  2.1   1.88 12.5   8.8   7.95  5.6   4.92  3.85  3.1
  3.55  2.25  1.95 10.74  9.3   7.    5.8   4.6   3.5   4.1 ]
[ 605000  465000

In [27]:
df_fh.to_csv(wd/'clean_aum.csv',index=False)

In [28]:
#6th cleaning
df_sip = pd.read_csv(rd/'04_monthly_sip_inflows.csv')
print(df_sip.head(24))
df_sip.dtypes

      month  sip_inflow_crore  active_sip_accounts_crore  \
0   2022-01             11517                       4.91   
1   2022-02             11438                       4.93   
2   2022-03             12328                       5.09   
3   2022-04             11863                       5.48   
4   2022-05             12286                       5.55   
5   2022-06             12276                       5.60   
6   2022-07             12140                       5.65   
7   2022-08             12694                       5.71   
8   2022-09             12976                       5.80   
9   2022-10             13040                       5.93   
10  2022-11             13306                       6.00   
11  2022-12             13573                       6.05   
12  2023-01             13856                       6.13   
13  2023-02             13687                       6.19   
14  2023-03             14276                       6.32   
15  2023-04             14749           

month                            str
sip_inflow_crore               int64
active_sip_accounts_crore    float64
new_sip_accounts_lakh        float64
sip_aum_lakh_crore           float64
yoy_growth_pct               float64
dtype: object

In [29]:
#validating at a glance
for cols in df_sip.columns:
    print(df_sip[cols].unique())

<StringArray>
['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07',
 '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-02',
 '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09',
 '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04',
 '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11',
 '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06',
 '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12']
Length: 48, dtype: str
[11517 11438 12328 11863 12286 12276 12140 12694 12976 13040 13306 13573
 13856 13687 14276 14749 14734 15245 15814 16042 16928 17073 17610 18838
 19187 20371 21262 23332 23547 24509 25323 25320 26459 26400 25999 25926
 26632 26688 27274 28464 28265 29361 29529 30200 31002]
[4.91 4.93 5.09 5.48 5.55 5.6  5.65 5.71 5.8  5.93 6.   6.05 6.13 6.19
 6.32 6.41 6.5  6.55 6.65 6.73 6.82 6.91 7.   7.1  7.2  7.3  7.4  7.6
 7.78 7.9  8

In [30]:
df_sip['yoy_growth_pct'].isna().sum()

np.int64(12)

In [31]:
df_sip = df_sip.fillna(0)

In [32]:
df_sip.head()

,month,sip_inflow_crore,active_sip_accounts_crore,new_sip_accounts_lakh,sip_aum_lakh_crore,yoy_growth_pct
0,2022-01,11517,4.91,9.10,4.80,0.0
1,2022-02,11438,4.93,8.20,4.85,0.0
2,2022-03,12328,5.09,10.50,5.01,0.0
3,2022-04,11863,5.48,9.52,5.12,0.0
4,2022-05,12286,5.55,8.10,5.15,0.0


In [33]:
df_sip['month'] = pd.to_datetime(df_sip['month'], format='%Y-%m') + pd.offsets.MonthEnd(0)
df_sip

,month,sip_inflow_crore,active_sip_accounts_crore,new_sip_accounts_lakh,sip_aum_lakh_crore,yoy_growth_pct
0,2022-01-31,11517,4.91,9.10,4.80,0.00
1,2022-02-28,11438,4.93,8.20,4.85,0.00
2,2022-03-31,12328,5.09,10.50,5.01,0.00
3,2022-04-30,11863,5.48,9.52,5.12,0.00
4,2022-05-31,12286,5.55,8.10,5.15,0.00
5,2022-06-30,12276,5.60,7.50,5.18,0.00
6,2022-07-31,12140,5.65,9.21,5.25,0.00
7,2022-08-31,12694,5.71,8.90,5.31,0.00
8,2022-09-30,12976,5.80,9.40,5.41,0.00
9,2022-10-31,13040,5.93,9.52,5.55,0.00


In [34]:
df_sip.to_csv(wd/'clean_sip.csv',index=False)

In [35]:
#7th cleaning
df_ci = pd.read_csv(rd/'05_category_inflows.csv')
print(df_ci.head(24))
df_ci.dtypes

      month           category  net_inflow_crore
0   2024-04          Large Cap            2413.0
1   2024-04            Mid Cap            3897.0
2   2024-04          Small Cap            3533.0
3   2024-04          Flexi Cap            4947.0
4   2024-04    Large & Mid Cap            4214.0
5   2024-04               ELSS             466.0
6   2024-04       Value/Contra            1328.0
7   2024-04  Sectoral/Thematic            8052.0
8   2024-04             Liquid           37537.0
9   2024-04     Short Duration            4400.0
10  2024-04               Gilt             784.0
11  2024-04             Hybrid            2955.0
12  2024-05          Large Cap            2076.0
13  2024-05            Mid Cap            5300.0
14  2024-05          Small Cap            4092.0
15  2024-05          Flexi Cap            5529.0
16  2024-05    Large & Mid Cap            4368.0
17  2024-05               ELSS             553.0
18  2024-05       Value/Contra            1361.0
19  2024-05  Sectora

month                   str
category                str
net_inflow_crore    float64
dtype: object

In [36]:
#validating at a glance
for cols in df_ci.columns:
    print(df_ci[cols].unique())

<StringArray>
['2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10',
 '2024-11', '2024-12', '2025-01', '2025-02', '2025-03']
Length: 12, dtype: str
<StringArray>
[        'Large Cap',           'Mid Cap',         'Small Cap',
         'Flexi Cap',   'Large & Mid Cap',              'ELSS',
      'Value/Contra', 'Sectoral/Thematic',            'Liquid',
    'Short Duration',              'Gilt',            'Hybrid']
Length: 12, dtype: str
[ 2413.  3897.  3533.  4947.  4214.   466.  1328.  8052. 37537.  4400.
   784.  2955.  2076.  5300.  4092.  5529.  4368.   553.  1361.  8354.
 41872.  4833.   836.  3487.  2519.  5047.  3535.  4478.  4610.   472.
  1386. 10030. 40486.  4321.   864.  3163.  2574.  4548.  3582.  4869.
  5023.   471.  1582.  9896. 34643.  4170.   959.  3291.  1940.  3899.
  3376.  5562.  5411.   499.  1308.  8360. 41952.  4658.   952.  3684.
  1879.  4960.  4137.  5397.  4528.   537.  1334.  8518. 35308.  5327.
   925.  3015.  2255.  4106.  4444.  60

In [37]:
df_ci['month'] = pd.to_datetime(df_ci['month'], format='%Y-%m') + pd.offsets.MonthEnd(0)
df_ci

,month,category,net_inflow_crore
0,2024-04-30,Large Cap,2413.0
1,2024-04-30,Mid Cap,3897.0
2,2024-04-30,Small Cap,3533.0
3,2024-04-30,Flexi Cap,4947.0
4,2024-04-30,Large & Mid Cap,4214.0
...,...,...,...
139,2025-03-31,Sectoral/Thematic,8614.0
140,2025-03-31,Liquid,38681.0
141,2025-03-31,Short Duration,4886.0
142,2025-03-31,Gilt,956.0


In [38]:
df_ci.to_csv(wd/'clean_category.csv',index=False)

In [39]:
#8th cleaning
df_ifc = pd.read_csv(rd/'06_industry_folio_count.csv')
print(df_ifc.head(24))
df_ifc.dtypes

      month  total_folios_crore  equity_folios_crore  debt_folios_crore  \
0   2022-01               13.26                 9.28               1.86   
1   2022-04               13.91                 9.74               1.95   
2   2022-07               13.85                 9.69               1.94   
3   2022-10               14.12                 9.88               1.98   
4   2023-01               14.81                10.37               2.07   
5   2023-04               15.54                10.88               2.18   
6   2023-07               16.28                11.40               2.28   
7   2023-10               16.72                11.70               2.34   
8   2024-01               17.78                12.45               2.49   
9   2024-04               18.85                13.20               2.64   
10  2024-07               19.98                13.99               2.80   
11  2024-10               21.62                15.13               3.03   
12  2024-12              

month                      str
total_folios_crore     float64
equity_folios_crore    float64
debt_folios_crore      float64
hybrid_folios_crore    float64
others_folios_crore    float64
dtype: object

In [40]:
df_ifc['month'] = pd.to_datetime(df_ifc['month'], format='%Y-%m') + pd.offsets.MonthEnd(0)
df_ifc

,month,total_folios_crore,equity_folios_crore,debt_folios_crore,hybrid_folios_crore,others_folios_crore
0,2022-01-31,13.26,9.28,1.86,0.80,1.33
1,2022-04-30,13.91,9.74,1.95,0.83,1.39
2,2022-07-31,13.85,9.69,1.94,0.83,1.38
3,2022-10-31,14.12,9.88,1.98,0.85,1.41
4,2023-01-31,14.81,10.37,2.07,0.89,1.48
5,2023-04-30,15.54,10.88,2.18,0.93,1.55
6,2023-07-31,16.28,11.40,2.28,0.98,1.63
7,2023-10-31,16.72,11.70,2.34,1.00,1.67
8,2024-01-31,17.78,12.45,2.49,1.07,1.78
9,2024-04-30,18.85,13.20,2.64,1.13,1.89


In [41]:
df_ifc.to_csv(wd/'clean_industry.csv',index=False)

In [42]:
#9th cleaning
df_ph = pd.read_csv(rd/'09_portfolio_holdings.csv')
print(df_ph.head(24))
df_ph.dtypes

    amfi_code stock_symbol                     stock_name       sector  \
0      119551    POWERGRID         Power Grid Corporation    Utilities   
1      119551     HDFCBANK                  HDFC Bank Ltd      Banking   
2      119551       GRASIM          Grasim Industries Ltd  Diversified   
3      119551      DRREDDY       Dr. Reddy's Laboratories       Pharma   
4      119551   ASIANPAINT               Asian Paints Ltd       Paints   
5      119551         NTPC                       NTPC Ltd    Utilities   
6      119551     DIVISLAB            Divi's Laboratories       Pharma   
7      119551   BHARTIARTL              Bharti Airtel Ltd      Telecom   
8      119551      HCLTECH           HCL Technologies Ltd           IT   
9      119551       MARUTI        Maruti Suzuki India Ltd   Automobile   
10     119552         INFY                    Infosys Ltd           IT   
11     119552    POWERGRID         Power Grid Corporation    Utilities   
12     119552      DRREDDY       Dr. R

amfi_code              int64
stock_symbol             str
stock_name               str
sector                   str
weight_pct           float64
market_value_cr      float64
current_price_inr    float64
portfolio_date           str
dtype: object

In [43]:
#validating at a glance
for cols in df_ph.columns:
    print(df_ph[cols].unique())

[119551 119552 119598 119599 100016 125497 100033 125498 120503 120504
 120505 120506 118632 118633 118634 118635 120841 120842 120843 119092
 119093 119094 119095 101206 101207 102885 102886 102887 148567 148568
 148569 149322 149323 149324]
<StringArray>
[ 'POWERGRID',   'HDFCBANK',     'GRASIM',    'DRREDDY', 'ASIANPAINT',
       'NTPC',   'DIVISLAB', 'BHARTIARTL',    'HCLTECH',     'MARUTI',
       'INFY',  'ICICIBANK',  'TATAMOTOR',      'CIPLA', 'BAJFINANCE',
  'SUNPHARMA', 'ULTRACEMCO',      'WIPRO',        'TCS',   'RELIANCE',
        'M&M',   'AXISBANK',  'KOTAKBANK',      'TITAN', 'HINDUNILVR',
 'INDUSINDBK',  'NESTLEIND',         'LT',       'SBIN',  'ADANIPORT']
Length: 30, dtype: str
<StringArray>
[       'Power Grid Corporation',                 'HDFC Bank Ltd',
         'Grasim Industries Ltd',      'Dr. Reddy's Laboratories',
              'Asian Paints Ltd',                      'NTPC Ltd',
           'Divi's Laboratories',             'Bharti Airtel Ltd',
          'H

In [44]:
df_ph.to_csv(wd/'clean_portfoilio.csv',index=False)

In [45]:
#10th cleaning
df_bi = pd.read_csv(rd/'10_benchmark_indices.csv')
print(df_bi.head(24))
df_bi.dtypes

          date index_name  close_value
0   2022-01-03    NIFTY50     17492.79
1   2022-01-04    NIFTY50     17689.64
2   2022-01-05    NIFTY50     17835.05
3   2022-01-06    NIFTY50     17878.51
4   2022-01-07    NIFTY50     17759.15
5   2022-01-10    NIFTY50     18124.84
6   2022-01-11    NIFTY50     18256.96
7   2022-01-12    NIFTY50     18162.54
8   2022-01-13    NIFTY50     18179.05
9   2022-01-14    NIFTY50     17971.75
10  2022-01-17    NIFTY50     17998.26
11  2022-01-18    NIFTY50     17938.46
12  2022-01-19    NIFTY50     18301.33
13  2022-01-20    NIFTY50     18424.66
14  2022-01-21    NIFTY50     18055.59
15  2022-01-24    NIFTY50     18338.54
16  2022-01-25    NIFTY50     18408.04
17  2022-01-26    NIFTY50     18650.54
18  2022-01-27    NIFTY50     18701.92
19  2022-01-28    NIFTY50     18734.13
20  2022-01-31    NIFTY50     18615.38
21  2022-02-01    NIFTY50     18724.17
22  2022-02-02    NIFTY50     18900.90
23  2022-02-03    NIFTY50     18640.35


date               str
index_name         str
close_value    float64
dtype: object

In [46]:
#validating at a glance
for cols in df_bi.columns:
    print(df_bi[cols].unique())

<StringArray>
['2022-01-03', '2022-01-04', '2022-01-05', '2022-01-06', '2022-01-07',
 '2022-01-10', '2022-01-11', '2022-01-12', '2022-01-13', '2022-01-14',
 ...
 '2026-05-18', '2026-05-19', '2026-05-20', '2026-05-21', '2026-05-22',
 '2026-05-25', '2026-05-26', '2026-05-27', '2026-05-28', '2026-05-29']
Length: 1150, dtype: str
<StringArray>
[        'NIFTY50',        'NIFTY100', 'NIFTY_MIDCAP150',    'BSE_SMALLCAP',
        'NIFTY500',   'CRISIL_LIQUID',     'CRISIL_GILT']
Length: 7, dtype: str
[17492.79 17689.64 17835.05 ...  2281.3   2298.35  2302.79]


In [47]:
df_bi['date']=pd.to_datetime(df_bi['date'])

In [48]:
df_bi[df_bi['close_value']<=0]

,date,index_name,close_value


In [49]:
df_bi.to_csv(wd/'clean_benchmark.csv',index=False)